# 1 — Loading California business data into Senzing

We have eight files of public California business data, from four different agencies:
company registrations, the people who run those companies, contractor licences,
alcohol licences and city business licences.

Because they come from different agencies, the same real company or person turns up
in several of them — spelled a little differently each time, with no shared id to
join on. **Entity resolution** is the job of working out which records describe the
same real-world thing, and that is what Senzing does.

In this notebook we write the code that turns each CSV row into a Senzing record and
load all of it. Notebook **02** then reads the results back out and reports on what
Senzing decided.

**Before you start**, from the project directory on your laptop:

```bash
docker compose up -d
./scripts/init_database.sh
```

---
## 1. Connect to Senzing

Senzing runs in its own container. We talk to it over gRPC, so all we need is that
container's address and a factory to create things from.

In [ ]:
import grpc
from senzing_grpc import SzAbstractFactoryGrpc

channel = grpc.insecure_channel("senzing:8261")
senzing = SzAbstractFactoryGrpc(channel)

A quick check that the connection works and that our licence covers this much data.

In [ ]:
import json

licence = json.loads(senzing.create_product().get_license())

print("licence type :", licence["licenseType"])
print("record limit :", f"{licence['recordLimit']:,}")
print("expires      :", licence["expireDate"])

---
## 2. The files

`docker-compose` mounts the project's `data` directory inside this container, so the
source files are right here.

In [ ]:
from pathlib import Path

DATA_DIR = Path("/workspace/data/filtered")

for path in sorted(DATA_DIR.iterdir()):
    print(f"{path.name:<52} {path.stat().st_size / 1_000_000:6.1f} MB")

---
## 3. Reading the files

Three of these files are ordinary CSV, one is tab separated, and three use a
delimiter you will not have seen before. We need a reader for each shape.

Start with the ordinary ones. We strip the whitespace around every value, because
several of these files pad their columns.

In [ ]:
import csv


def read_table(path, delimiter=","):
    """Read a normal delimited file, trimming the padding around every value."""
    with open(path, newline="", encoding="utf-8", errors="replace") as handle:
        for row in csv.DictReader(handle, delimiter=delimiter):
            yield {key: (value or "").strip() for key, value in row.items()}


def read_tsv(path):
    return read_table(path, delimiter="\t")

The three Secretary of State files separate their fields with the literal three
characters `*|*`, and quote nothing. That is not a CSV dialect, so `csv` cannot help
us — we split the lines ourselves.

In [ ]:
def read_sos(path):
    """Read a Secretary of State file. Fields are separated by the string *|*."""
    lines = Path(path).read_text(encoding="utf-8", errors="replace").splitlines()
    header = lines[0].split("*|*")
    for line in lines[1:]:
        if line.strip():
            yield dict(zip(header, line.split("*|*")))

The contractor personnel file needs one special rule. Its `Name` column packs a
person's name into fixed-width sub-columns, so if we trim it the column positions all
shift. Every other value gets trimmed as usual.

In [ ]:
def read_personnel(path):
    """Like read_table, but Name keeps its padding — its sub-columns are fixed width."""
    with open(path, newline="", encoding="utf-8", errors="replace") as handle:
        for row in csv.DictReader(handle):
            yield {
                key: (value or "") if key == "Name" else (value or "").strip()
                for key, value in row.items()
            }

Here is one row from the contractor licence file — the first 13 of its 52 columns.

In [ ]:
contractor_rows = read_table(DATA_DIR / "master-list-of-ca-licensed-contractors.csv")
raw_contractor = next(iter(contractor_rows))

{key: raw_contractor[key] for key in list(raw_contractor)[:13]}

---
## 4. What we are aiming for

A Senzing record is a small, flat piece of JSON. It looks like this:

```json
{
  "DATA_SOURCE": "CSLB_CONTRACTORS",
  "RECORD_ID": "1000016",
  "FEATURES": [
    {"RECORD_TYPE": "ORGANIZATION"},
    {"NAME_TYPE": "PRIMARY", "NAME_ORG": "STEALTH LEAK SPECIALISTS INC"},
    {"ADDR_TYPE": "BUSINESS", "ADDR_LINE1": "500 ANVILWOOD DRIVE", "ADDR_CITY": "OAKLEY"}
  ],
  "COUNTY": "Contra Costa"
}
```

Four things are going on:

- **`DATA_SOURCE`** — which file this came from. **`RECORD_ID`** — its id inside that
  file. Together they are unique, and they are how we ask for this record later.
- **`FEATURES`** — the things Senzing **matches on**. One entry per name, address,
  phone or identifier.
- Anything **outside `FEATURES`** is payload: carried along so we can read it later,
  but never used for matching. `COUNTY` above is payload.
- Senzing has its **own attribute names**. A business name is `NAME_ORG`; a person's
  surname is `NAME_LAST`; a phone number is `PHONE_NUMBER`. Making up your own names
  here is the single most common way to get bad results — the load succeeds and the
  matching quietly does nothing.

Our job is to get from the 52-column row above to that.

---
## 5. Six small helpers

Almost every column we care about becomes a name, an address, a phone number or a
relationship. So rather than writing the same dictionary over and over, we write one
tiny function per kind of thing.

The first one does the only clever bit in this whole notebook. Our source files are
full of empty columns, and Senzing should not be told about a blank address. So
`feature` drops anything empty — and if all that survives are the `*_TYPE` labels
(which describe a value rather than being one), it drops the whole feature.

That means the mapping code below can pass every column straight in without checking
whether it is populated first.

In [ ]:
def feature(**attributes):
    """One entry for a record's FEATURES list, or None if there is nothing in it."""
    values = {key: value for key, value in attributes.items() if value}
    has_a_value = any(not key.endswith("_TYPE") for key in values)
    return values if has_a_value else None

Now names. Senzing keeps people and organisations strictly apart: an organisation
gets `NAME_ORG`, a person gets `NAME_FIRST` / `NAME_MIDDLE` / `NAME_LAST`, and a
single record must never carry both kinds.

`NAME_TYPE` says which name this is — `PRIMARY` for the real one, `AKA` for a trading
name or alias. A record can have several names.

In [ ]:
def org_name(value, kind="PRIMARY"):
    return feature(NAME_TYPE=kind, NAME_ORG=value)


def person_name(first="", middle="", last="", suffix="", kind="PRIMARY"):
    return feature(
        NAME_TYPE=kind,
        NAME_FIRST=first,
        NAME_MIDDLE=middle,
        NAME_LAST=last,
        NAME_SUFFIX=suffix,
    )

Addresses and phones. Where a file gives us the address already broken into columns
we hand Senzing the parts; where a file gives us one lump of text we hand it
`ADDR_FULL` and let Senzing parse it. Never split an address string yourself.

`ADDR_TYPE="BUSINESS"` is worth setting on companies — Senzing gives an
organisation's business address extra weight when matching.

In [ ]:
def address(kind, line1="", line2="", line3="", city="", state="", postal_code="", country=""):
    """An address from separate columns. Blank parts drop out."""
    return feature(
        ADDR_TYPE=kind,
        ADDR_LINE1=line1,
        ADDR_LINE2=line2,
        ADDR_LINE3=line3,
        ADDR_CITY=city,
        ADDR_STATE=state,
        ADDR_POSTAL_CODE=postal_code,
        ADDR_COUNTRY=country,
    )


def full_address(kind, text):
    """An address that arrived as one string. Senzing parses it, so we do not."""
    return feature(ADDR_TYPE=kind, ADDR_FULL=text)


def phone(number):
    return feature(PHONE_TYPE="BUSINESS", PHONE_NUMBER=number)

`RECORD_TYPE` tells Senzing whether this record is a `PERSON` or an `ORGANIZATION`.
It matters more than it looks: **Senzing will never merge a PERSON with an
ORGANIZATION**, however well the names and addresses agree.

The last two helpers record relationships the *source told us about* — as opposed to
matches Senzing works out for itself. A record that others may point at gets an
**anchor**; a record that points at one gets a **pointer** naming the role.

In [ ]:
def record_type(kind):
    return {"RECORD_TYPE": kind}


def anchor(domain, key):
    """Mark this record as something other records are allowed to point at."""
    return {"REL_ANCHOR_DOMAIN": domain, "REL_ANCHOR_KEY": key}


def points_at(domain, key, role):
    """Record a relationship from this record to an anchored one."""
    return {"REL_POINTER_DOMAIN": domain, "REL_POINTER_KEY": key, "REL_POINTER_ROLE": role}

Finally, the function that puts a whole record together.

In [ ]:
def build_record(data_source, record_id, features, **payload):
    """Assemble a Senzing record, dropping empty features and empty payload values."""
    return {
        "DATA_SOURCE": data_source,
        "RECORD_ID": record_id,
        "FEATURES": [item for item in features if item],
        **{key: value for key, value in payload.items() if value},
    }

---
## 6. Mapping the contractor file

Now the actual mapping. Out of 52 columns we keep eight as features: three names, an
address, and a phone number. The licence number becomes the `RECORD_ID` — it is not
a feature, because it identifies the *row*, not the business.

Eight more columns become payload. The rest we deliberately drop, and it is worth
saying why: they are bond and insurance-policy details. A bond number identifies a
*bond*, not a business, and the surety company's name is somebody else's name. Put
those in as features and you would resolve unrelated contractors together because
they happen to share an insurer.

In [ ]:
def map_contractor(row):
    """A business holding a CSLB contractor licence."""
    return build_record(
        "CSLB_CONTRACTORS",
        row["LicenseNo"],
        [
            record_type("ORGANIZATION"),
            org_name(row["BusinessName"]),
            org_name(row["BUS-NAME-2"], "AKA"),
            org_name(row["FullBusinessName"], "AKA"),
            address("BUSINESS", line1=row["MailingAddress"], city=row["City"],
                    state=row["State"], postal_code=row["ZIPCode"]),
            phone(row["BusinessPhone"]),
            anchor("CSLB_CONTRACTORS", row["LicenseNo"]),
        ],
        COUNTY=row["County"],
        BUSINESS_FORM=row["BusinessType"],
        PRIMARY_STATUS=row["PrimaryStatus"],
        SECONDARY_STATUS=row["SecondaryStatus"],
        CLASSIFICATIONS=row["Classifications(s)"],
        ISSUE_DATE=row["IssueDate"],
        EXPIRATION_DATE=row["ExpirationDate"],
        WORKERS_COMP_COVERAGE=row["WorkersCompCoverageType"],
    )

In [ ]:
print(json.dumps(map_contractor(raw_contractor), indent=2))

Notice what `feature` did for us: this business has no second or third name and no
`ADDR_LINE2`, so those never appear. We passed the blank columns in anyway and they
quietly disappeared.

---
## 7. Mapping the people

The personnel file lists the people behind each contractor licence. Its `Name` column
is the awkward one — look at the raw value:

In [ ]:
personnel_rows = read_personnel(DATA_DIR / "master-list-of-ca-licensed-contractors-personnel.csv")
raw_person = next(iter(personnel_rows))

repr(raw_person["Name"])

That is not one name with spaces in it. It is four fixed-width columns —
surname, first name, middle name, then a suffix column holding things like `JR` and
`III` — packed into a single field. We slice them apart by character position.

(How do we know? We profiled the file: across all 80,972 person names, not one
crosses a column boundary.)

In [ ]:
NAME_COLUMNS = {"last": (1, 36), "first": (36, 51), "middle": (51, 63), "suffix": (63, None)}


def split_name(segment):
    """Pull the four fixed-width name columns out of one Name value."""
    return {part: segment[start:end].strip() for part, (start, end) in NAME_COLUMNS.items()}


split_name(raw_person["Name"])

Two more wrinkles in this file:

- The `Name` column can hold **two** names separated by `|` — an alias, or the second
  company in a joint venture. So we loop over the pieces.
- Not every row is a person. `Name-TP` tells us: rows starting `Principal` are people,
  rows starting `Business` are companies. That decides both the `RECORD_TYPE` and
  which name helper we use.

In [ ]:
def map_personnel(row):
    """A person, or a partner company, attached to a contractor licence."""
    is_person = row["Name-TP"].split("|")[0].strip() == "Principal"

    names = []
    for position, segment in enumerate(row["Name"].split("|")):
        kind = "PRIMARY" if position == 0 else "AKA"
        if is_person:
            names.append(person_name(**split_name(segment), kind=kind))
        else:
            names.append(org_name(segment.strip(), kind))

    return build_record(
        "CSLB_PERSONNEL",
        f'{row["LIC-NO"]}-{row["SEQ-NO"]}',
        [
            record_type("PERSON" if is_person else "ORGANIZATION"),
            *names,
            points_at("CSLB_CONTRACTORS", row["LIC-NO"], "PRINCIPAL_OF"),
        ],
        NAME_KIND=row["Name-TP"],
        JOB_TITLE=row["EMP-Titl-CDE"],
        SURETY_KIND=row["SURETY-TP"],
        SURETY_COMPANY=row["SuretyCompany"],
        BOND_NUMBER=row["BOND-NO"],
        BOND_AMOUNT=row["BOND-AMT"],
    )

In [ ]:
print(json.dumps(map_personnel(raw_person), indent=2))

This record is a `PERSON`, its name arrived as separate parts, and it
`PRINCIPAL_OF`-points at contractor licence `22` — the same licence number that
appears in a contractor record's anchor. That is how the two files get tied together
without Senzing having to guess.

---
## 8. The remaining six files

Same ideas, so these go quickly. The Secretary of State filings are the richest: each
company can carry three addresses, and we keep all of them.

In [ ]:
def map_filing(row):
    """A business registered with the CA Secretary of State."""
    return build_record(
        "CA_SOS_FILINGS",
        row["ENTITY_NUM"],
        [
            record_type("ORGANIZATION"),
            org_name(row["ENTITY_NAME"]),
            org_name(row["FOREIGN_NAME"], "AKA"),
            feature(REGISTRATION_DATE=row["INITIAL_FILING_DATE"]),
            address("BUSINESS", line1=row["PRINCIPAL_ADDRESS"], line2=row["PRINCIPAL_ADDRESS2"],
                    city=row["PRINCIPAL_CITY"], state=row["PRINCIPAL_STATE"],
                    postal_code=row["PRINCIPAL_POSTAL_CODE"], country=row["PRINCIPAL_COUNTRY"]),
            address("MAILING", line1=row["MAILING_ADDRESS"], line2=row["MAILING_ADDRESS2"],
                    line3=row["MAILING_ADDRESS3"], city=row["MAILING_CITY"],
                    state=row["MAILING_STATE"], postal_code=row["MAILING_POSTAL_CODE"],
                    country=row["MAILING_COUNTRY"]),
            address("BUSINESS", line1=row["PRINCIPAL_ADDRESS_IN_CA"],
                    line2=row["PRINCIPAL_ADDRESS2_IN_CA"], city=row["PRINCIPAL_CITY_IN_CA"],
                    state=row["PRINCIPAL_STATE_IN_CA"],
                    postal_code=row["PRINCIPAL_POSTAL_CODE_IN_CA"],
                    country=row["PRINCIPAL_COUNTRY_IN_CA"]),
            anchor("CA_SOS_FILINGS", row["ENTITY_NUM"]),
        ],
        ENTITY_STATUS=row["ENTITY_STATUS"],
        ENTITY_TYPE=row["ENTITY_TYPE"],
        FILING_TYPE=row["FILING_TYPE"],
        JURISDICTION=row["JURISDICTION"],
    )

Every filing names an **agent for service of process**. Sometimes that is a person,
sometimes a company that does this for a living — and `AGENT_TYPE` says which, so we
let it choose the record type and the name helper.

In [ ]:
def map_agent(row):
    """The agent for service of process named on a filing."""
    is_person = row["AGENT_TYPE"] == "Individual Agent"

    return build_record(
        "CA_SOS_AGENTS",
        row["ENTITY_NUM"],
        [
            record_type("PERSON" if is_person else "ORGANIZATION"),
            person_name(first=row["FIRST_NAME"], middle=row["MIDDLE_NAME"], last=row["LAST_NAME"])
            if is_person
            else org_name(row["ORG_NAME"]),
            address("BUSINESS", line1=row["PHYSICAL_ADDRESS1"], line2=row["PHYSICAL_ADDRESS2"],
                    line3=row["PHYSICAL_ADDRESS3"], city=row["PHYSICAL_CITY"],
                    state=row["PHYSICAL_STATE"], postal_code=row["PHYSICAL_POSTAL_CODE"],
                    country=row["PHYSICAL_COUNTRY"]),
            points_at("CA_SOS_FILINGS", row["ENTITY_NUM"], "AGENT_OF"),
        ],
        AGENT_TYPE=row["AGENT_TYPE"],
    )

Principals are the officers, directors and managers. A few of them are companies
rather than people — we can tell because the `ORG_NAME` column is filled in.

This file has no id column of its own, so we build a `RECORD_ID` out of the things
that make a principal distinct: which company, what role, their name and their
address. The address matters — the same person is sometimes listed twice for one
company at two different addresses, and those are genuinely two rows to keep.

In [ ]:
def principal_record_id(row):
    parts = [row["ENTITY_NUM"], row["POSITION_TYPE"], row["ORG_NAME"], row["FIRST_NAME"],
             row["MIDDLE_NAME"], row["LAST_NAME"], row["ADDRESS1"], row["ADDRESS2"]]
    return "-".join(part for part in parts if part)


def map_principal(row):
    """An officer, director or manager of a registered business."""
    is_organization = bool(row["ORG_NAME"])

    return build_record(
        "CA_SOS_PRINCIPALS",
        principal_record_id(row),
        [
            record_type("ORGANIZATION" if is_organization else "PERSON"),
            org_name(row["ORG_NAME"])
            if is_organization
            else person_name(first=row["FIRST_NAME"], middle=row["MIDDLE_NAME"],
                             last=row["LAST_NAME"]),
            address("BUSINESS", line1=row["ADDRESS1"], line2=row["ADDRESS2"],
                    line3=row["ADDRESS3"], city=row["CITY"], state=row["STATE"],
                    postal_code=row["POSTAL_CODE"], country=row["COUNTRY"]),
            points_at("CA_SOS_FILINGS", row["ENTITY_NUM"], "PRINCIPAL_OF"),
        ],
        POSITION_TYPE=row["POSITION_TYPE"],
    )

The alcohol licences need one piece of cleaning up. Their address column has the
census tract glued onto the end of the ZIP code with no separator:

```
272 S MAPLE AVE,SOUTH SAN FRANCISCO, CA  94080Census Tract:  6023.00
```

Leave that in and the ZIP reads as `94080Census`, so we cut it off.

The licence number here *is* worth keeping as a feature. One business often holds
several licences of different types, so the number appears on several rows — and
because it identifies the licensee rather than the row, it is a genuine clue that
those rows are the same business. It goes in as `OTHER_ID`, which is what Senzing
uses for an identifier that has no more specific home.

In [ ]:
def premises_address(text):
    """Drop the census tract glued onto the end of an ABC premises address."""
    return text.split("Census Tract:")[0].strip()


def map_abc_licence(row, data_source):
    """An alcohol licence from CA Alcoholic Beverage Control."""
    trade_name = row["Business Name"] if row["Business Name"] != "NA" else ""

    return build_record(
        data_source,
        f'{row["License Number"]}-{row["License Type"]}',
        [
            record_type("ORGANIZATION"),
            org_name(row["Primary Owner"]),
            org_name(trade_name, "AKA"),
            feature(OTHER_ID_TYPE="CA_ABC_LICENSE", OTHER_ID_NUMBER=row["License Number"]),
            feature(REGISTRATION_DATE=row["Orig. Iss. Date"]),
            full_address("BUSINESS", premises_address(row["Premises Addr."])),
        ],
        LICENCE_STATUS=row["Status"],
        LICENCE_CLASS=row["License Type"],
        EXPIRATION_DATE=row["Expir. Date"],
        GEO_CODE=row["Geo Code"],
    )


def map_abc_retail(row):
    return map_abc_licence(row, "CA_ABC_RETAIL")


def map_abc_nonretail(row):
    return map_abc_licence(row, "CA_ABC_NONRETAIL")

Last one, and the simplest. Burlingame's city licences have five columns. The address
arrives as a single string, so it goes in as `ADDR_FULL` — except where the city has
withheld it and written `--ON FILE--` instead, which is a placeholder, not an address.

In [ ]:
def map_burlingame(row):
    """A City of Burlingame business licence."""
    withheld = row["Address"] == "--ON FILE--"

    return build_record(
        "BURLINGAME_LICENSES",
        row["Account #"],
        [
            record_type("ORGANIZATION"),
            org_name(row["Business Name"]),
            full_address("BUSINESS", "" if withheld else row["Address"]),
            feature(REGISTRATION_DATE=row["Start Date"]),
        ],
        EXPIRATION_DATE=row["Expire Date"],
    )

---
## 9. Map everything

Eight files, each with a reader and a mapping function. That is the whole pipeline:

In [ ]:
SOURCES = [
    ("Filings.csv", read_sos, map_filing),
    ("Agents.csv", read_sos, map_agent),
    ("Principals.csv", read_sos, map_principal),
    ("master-list-of-ca-licensed-contractors.csv", read_table, map_contractor),
    ("master-list-of-ca-licensed-contractors-personnel.csv", read_personnel, map_personnel),
    ("CA-ABC-LicenseReport-retail.csv", read_table, map_abc_retail),
    ("CA-ABC-LicenseReport-nonretail.csv", read_table, map_abc_nonretail),
    ("burlingame.tsv", read_tsv, map_burlingame),
]

In [ ]:
records = []

for filename, read_rows, map_row in SOURCES:
    for row in read_rows(DATA_DIR / filename):
        records.append(map_row(row))

print(f"{len(records):,} records")

In [ ]:
import collections

collections.Counter(record["DATA_SOURCE"] for record in records).most_common()

A quick sanity check before we load anything: `DATA_SOURCE` and `RECORD_ID` together
have to be unique, or records will silently overwrite each other.

In [ ]:
keys = {(record["DATA_SOURCE"], record["RECORD_ID"]) for record in records}

print(f"{len(keys):,} unique keys for {len(records):,} records")

---
## 10. Register the data sources

Senzing will not accept a record until it knows the name of the data source it came
from. Registering means editing Senzing's configuration: take the current one, add
our names, save it back as the new default. Re-running this is safe.

In [ ]:
DATA_SOURCES = sorted({record["DATA_SOURCE"] for record in records})

config_manager = senzing.create_configmanager()

old_config_id = config_manager.get_default_config_id()
config = config_manager.create_config_from_config_id(old_config_id)

for name in DATA_SOURCES:
    config.register_data_source(name)

new_config_id = config_manager.register_config(config.export(), "ODSC workshop data sources")
config_manager.replace_default_config_id(old_config_id, new_config_id)

print("new configuration id:", new_config_id)

The engine is still holding the old configuration, so point it at the new one.

In [ ]:
senzing.reinitialize(config_manager.get_default_config_id())
engine = senzing.create_engine()

---
## 11. Load

`add_record` sends one record. Senzing compares it against everything already loaded
and decides whether it is something new or belongs with an entity it already has.

Two practical details:

- We use a **thread pool**. One record at a time takes about half an hour; eight
  threads brings it under ten minutes.
- When two threads touch the same entity at once, Senzing raises `SzRetryableError`.
  The name says it all — wait a moment and try again.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

from senzing import SzRetryableError


def add_record(record):
    """Send one record, backing off if another thread has that entity locked."""
    for attempt in range(6):
        try:
            engine.add_record(record["DATA_SOURCE"], record["RECORD_ID"], json.dumps(record))
            return True
        except SzRetryableError:
            time.sleep(0.5 * (attempt + 1))
    return False

**This cell takes about 10 minutes.** It prints progress as it goes.

In [ ]:
started = time.time()
failed = 0

with ThreadPoolExecutor(max_workers=8) as pool:
    for done, succeeded in enumerate(pool.map(add_record, records), start=1):
        if not succeeded:
            failed += 1
        if done % 25_000 == 0:
            print(f"{done:>7,} / {len(records):,}   ({time.time() - started:.0f}s)")

print(f"\nloaded {len(records):,} records in {(time.time() - started) / 60:.1f} minutes")
print(f"failed: {failed}")

---
## 12. Finish the follow-up work

While loading, Senzing queues **redo records**: second thoughts to revisit once more
data has arrived. A record loaded early might match something loaded much later, and
this is how that gets picked up.

Working through the queue is what makes the answers final, so it is not optional —
and notebook 02 would report the wrong numbers if we skipped it.

In [ ]:
print(f"{engine.count_redo_records():,} redo records waiting")

**This cell takes about 3 minutes.**

In [ ]:
started = time.time()
processed = 0

while True:
    redo_record = engine.get_redo_record()
    if not redo_record:
        break
    engine.process_redo_record(redo_record)
    processed += 1
    if processed % 4_000 == 0:
        print(f"{processed:>7,} processed, {engine.count_redo_records():,} left")

print(f"\nprocessed {processed:,} redo records in {(time.time() - started) / 60:.1f} minutes")
print("remaining:", engine.count_redo_records())

---
## 13. One quick look before we move on

Everything is in. As a sanity check, ask for the entity that one contractor record
ended up in, and see what else is in there with it.

In [ ]:
entity = json.loads(engine.get_entity_by_record_id("CSLB_CONTRACTORS", "1000087"))

print(entity["RESOLVED_ENTITY"]["ENTITY_NAME"])
for member in entity["RESOLVED_ENTITY"]["RECORDS"]:
    print(f"   {member['DATA_SOURCE']:<22} {member['RECORD_ID']}")

Two records, from two different agencies, are now one entity. Nothing in either file
said they were the same business — no shared id, no common key. Senzing worked it out
from the name and the address.

That is one entity out of many. **Notebook 02** answers the real question: across all
153,094 records, how many entities are there, how much did they overlap, and can we
trust the result?